# Facility Location Problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frankhuettner/ManSciDA-Excel-to-app/blob/main/facility-location-app/facility_location.ipynb)

**Model:** Mixed Integer Linear Program (MILP)

$$\min_{y, x} \quad \sum_j f_j y_j + \sum_{i,j} c_{ij} x_{ij}$$
$$\text{s.t.} \quad \sum_j x_{ij} \geq d_i \quad (\text{demand}), \quad \sum_i x_{ij} \leq \text{cap}_j \cdot y_j \quad (\text{capacity}), \quad y_j \in \{0,1\}, \quad x_{ij} \geq 0$$

**Solver:** `scipy.optimize.milp` (HiGHS)  
**Same code** runs in the browser app (`facility-location-app/index.html`) via Pyodide.

---
### Environment
| Context | Runtime | Packages | Install |
|---|---|---|---|
| This notebook | Google Colab / local Python (uv) | `scipy`, `numpy` | pre-installed in Colab / `uv sync` |
| Browser app | Pyodide (WebAssembly) | `scipy`, `numpy` | `pyodide.loadPackage(["numpy", "scipy"])` |

Package names are **identical** in this case — scipy is in both PyPI and Pyodide's index.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
import scipy

print(f"scipy  {scipy.__version__}")
print(f"numpy  {np.__version__}")

## Data
Source: `facility-location-problem.xlsx`

In [ ]:
FACILITIES = ["West", "North", "East"]
CUSTOMERS  = ["R1",   "R2",   "R3"]
BIG_M = 1e5   # marks an unavailable route (same convention as in the Excel file)

# Variable transport cost c[i][j]: customer i served by facility j
c_mat = np.array([
    [1.0,   0.4,   2.4],   # R1 ← West, North, East
    [BIG_M, 0.5,   0.8],   # R2 ← West unavailable
    [1.3,   BIG_M, 1.4],   # R3 ← North unavailable
])

f   = np.array([80.,  150., 100.])   # fixed opening cost per facility
cap = np.array([300., 100., 200.])   # capacity per facility
d   = np.array([100.,  60., 240.])   # demand per customer

pd.DataFrame(c_mat, index=CUSTOMERS, columns=FACILITIES)

## MILP formulation

Variable layout (flat vector):
```
[ y_West, y_North, y_East,
  x_R1_West, x_R1_North, x_R1_East,
  x_R2_West, x_R2_North, x_R2_East,
  x_R3_West, x_R3_North, x_R3_East ]
```
`x[i,j]` index = `n_f + i*n_f + j`

`scipy.optimize.milp` expects `lb ≤ A @ x ≤ ub` for all constraints.

In [ ]:
def solve_facility_location(c_mat, f, cap, d):
    """Identical to the _solve() function embedded in index.html."""
    n_f, n_c = len(f), len(d)
    nv = n_f + n_c * n_f   # 3 binary + 9 continuous = 12 variables

    # Objective: fixed costs + transport costs
    obj = np.zeros(nv)
    obj[:n_f] = f
    for i in range(n_c):
        obj[n_f + i*n_f : n_f + (i+1)*n_f] = c_mat[i]

    rows, lo, hi = [], [], []

    # Demand:   sum_j x[i,j]  >=  d[i]
    for i in range(n_c):
        r = np.zeros(nv)
        r[n_f + i*n_f : n_f + (i+1)*n_f] = 1.0
        rows.append(r); lo.append(d[i]); hi.append(np.inf)

    # Capacity: sum_i x[i,j]  <=  cap[j] * y[j]
    #           --> sum_i x[i,j] - cap[j]*y[j]  <=  0
    for j in range(n_f):
        r = np.zeros(nv); r[j] = -cap[j]
        for i in range(n_c):
            r[n_f + i*n_f + j] = 1.0
        rows.append(r); lo.append(-np.inf); hi.append(0.0)

    lb = np.zeros(nv)
    ub = np.full(nv, np.inf); ub[:n_f] = 1.0   # y_j ∈ {0,1}
    integ = np.zeros(nv);     integ[:n_f] = 1   # y_j binary

    res = milp(
        obj,
        constraints=LinearConstraint(np.array(rows), lo, hi),
        integrality=integ,
        bounds=Bounds(lb, ub),
    )
    if res.status != 0:
        raise RuntimeError(f"Solver failed: {res.message}")

    y = np.round(res.x[:n_f]).astype(int)
    x = res.x[n_f:].reshape(n_c, n_f)
    return y, x, res.fun

## Solve and test

In [ ]:
y, x, total = solve_facility_location(c_mat, f, cap, d)

open_facs = [FACILITIES[j] for j in range(3) if y[j]]
fixed_cost = float((f * y).sum())
mask       = c_mat < BIG_M * 0.9
var_cost   = float((c_mat * x * mask).sum())

print(f"Open facilities : {open_facs}")
print(f"Fixed cost      : {fixed_cost:.2f}")
print(f"Variable cost   : {var_cost:.2f}")
print(f"Total cost      : {total:.2f}")

# ── Excel reference (baseline: Excel Solver, default data) ──
EXCEL_OPEN      = [1, 0, 1]  # West open, North closed, East open
EXCEL_COST      = 644.0      # total cost
EXCEL_x_R1_West = 100.0      # R1 demand fully served by West
EXCEL_x_R2_East =  60.0      # R2 demand fully served by East

# Assertions against Excel reference
assert list(y)                         == EXCEL_OPEN,  f"Facilities: {y}"
assert abs(total - EXCEL_COST)         < 0.01,         f"Total cost: {total}"
assert abs(x[0, 0] - EXCEL_x_R1_West) < 0.01,         "R1 → West"
assert abs(x[1, 2] - EXCEL_x_R2_East) < 0.01,         "R2 → East"

print("✓ All assertions pass")

## Results

In [ ]:
# Flow matrix
flow_df = pd.DataFrame(x.round(2), index=CUSTOMERS, columns=FACILITIES)
flow_df["Total served"] = flow_df.sum(axis=1)
flow_df["Demand"]       = d.astype(int)

usage_row = pd.Series(x.sum(axis=0).tolist() + [None, None],
                      index=flow_df.columns, name="Usage")
cap_row   = pd.Series(cap.tolist() + [None, None],
                      index=flow_df.columns, name="Capacity")

display(pd.concat([flow_df, usage_row.to_frame().T, cap_row.to_frame().T]))

# Cost summary
display(pd.DataFrame({
    "Item":   ["Fixed cost", "Variable cost", "Total cost"],
    "Amount": [fixed_cost, var_cost, total],
}).set_index("Item"))

# Facility decisions
display(pd.DataFrame({
    "Facility": FACILITIES,
    "Open":     [bool(yj) for yj in y],
    "Fixed cost (if open)": f.tolist(),
    "Capacity":  cap.tolist(),
    "Usage":     x.sum(axis=0).tolist(),
}).set_index("Facility"))

## Sensitivity: what if demands change?

Re-solve for a range of R3 demand values to see when North must open.

In [ ]:
results = []
for d3 in range(100, 420, 20):
    d_new = np.array([100., 60., float(d3)])
    try:
        y_s, _, cost_s = solve_facility_location(c_mat, f, cap, d_new)
        results.append({"d_R3": d3, "West": bool(y_s[0]), "North": bool(y_s[1]),
                        "East": bool(y_s[2]), "Total cost": round(cost_s, 2)})
    except RuntimeError:
        results.append({"d_R3": d3, "West": None, "North": None,
                        "East": None, "Total cost": "infeasible"})

pd.DataFrame(results).set_index("d_R3")